### Setup


In [1]:
# Add the parent directory of the current working directory to the Python path at runtime. 
# In order to import modules from the src directory.
import os
import sys 

current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)

In [2]:
import numpy as np
import pandas as pd
import bambi as bmb
import arviz as az

from prettytable import PrettyTable

from src.stat_utils import *
from src.anl_utils import load_data

### Load and prepare data

In [3]:
sim_results_folder = '../results/simulation'
data_folder = '../data'
sync_at_file = os.path.join(sim_results_folder, 'first_session_arnold_tongues.npy')
emp_at_file = os.path.join(data_folder, 'Experiment.csv')

In [5]:
# Load the simulations results and the empirical data
sync_results = np.load(sync_at_file)
sync_results_vector = sync_results.mean(axis=0).flatten()
data = load_data(emp_at_file)

# Dummy code session 9
data['Transfer'] = (data['SessionID'] == 9)

# Filter data for learning phase (Sessions 1 to 8)
data_learning = data[data['SessionID'] <= 8].copy()

# Map synchrony values to each condition in DataFrame
data_learning['Synchrony'] = data_learning['Condition'].apply(lambda x: sync_results_vector[x-1])

# Z-score the relevant columns
data_learning = zscore_data(data_learning, ['ContrastHeterogeneity', 'GridCoarseness', 'Synchrony'])
data = zscore_data(data, ['ContrastHeterogeneity', 'GridCoarseness'])

# Center the session number
data_learning['SessionID'] = data_learning['SessionID'] - data_learning['SessionID'].mean()
data['SessionID'] = data['SessionID'] - data['SessionID'].mean()

### Define statistical models

In [11]:
model_features = bmb.Model(
    "Correct ~ 1 + ContrastHeterogeneity * GridCoarseness + SessionID * (ContrastHeterogeneity + GridCoarseness) +  (1 + SessionID + ContrastHeterogeneity * GridCoarseness|SubjectID)",
    data=data,
    family="bernoulli"
)

Does learning occur (i.e., does session have an effect on performance)? 

Do the effects of contrast heterogeneity and/or grid coarseness depend on session?

In [12]:
idata_features = model_features.fit(
    draws=2000, tune=2000, target_accept=0.9,
    idata_kwargs={"log_likelihood": True}, progressbar=False
)

Modeling the probability that Correct==1
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [Intercept, ContrastHeterogeneity, GridCoarseness, ContrastHeterogeneity:GridCoarseness, SessionID, SessionID:ContrastHeterogeneity, SessionID:GridCoarseness, 1|SubjectID_sigma, 1|SubjectID_offset, SessionID|SubjectID_sigma, SessionID|SubjectID_offset, ContrastHeterogeneity|SubjectID_sigma, ContrastHeterogeneity|SubjectID_offset, GridCoarseness|SubjectID_sigma, GridCoarseness|SubjectID_offset, ContrastHeterogeneity:GridCoarseness|SubjectID_sigma, ContrastHeterogeneity:GridCoarseness|SubjectID_offset]
Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 12006 seconds.
There were 52 divergences after tuning. Increase `target_accept` or reparameterize.


In [15]:
predictors = ["SessionID", "ContrastHeterogeneity", "GridCoarseness", "SessionID:ContrastHeterogeneity","SessionID:GridCoarseness", "ContrastHeterogeneity:GridCoarseness"]
directions = ['greater', 'less', 'less','less','greater', 'greater']

posterior = posterior_table(idata_features, predictors, directions)

odds_ratios = OR_table(idata_features, predictors)

print("One-sided posterior probabilities:")
print(posterior)
print("\nOdds ratios:")
print(odds_ratios)

az.summary(idata_features, var_names=predictors, hdi_prob=0.95)

One-sided posterior probabilities:
+--------------------------------------+-----------+-------+
|              Predictor               | direction |   P   |
+--------------------------------------+-----------+-------+
|              SessionID               |  greater  | 1.000 |
|        ContrastHeterogeneity         |    less   | 1.000 |
|            GridCoarseness            |    less   | 1.000 |
|   SessionID:ContrastHeterogeneity    |    less   | 1.000 |
|       SessionID:GridCoarseness       |  greater  | 0.410 |
| ContrastHeterogeneity:GridCoarseness |  greater  | 1.000 |
+--------------------------------------+-----------+-------+

Odds ratios:
+--------------------------------------+--------+--------------+---------------+
|              Predictor               |  Mean  | Lower (2.5%) | Upper (97.5%) |
+--------------------------------------+--------+--------------+---------------+
|              SessionID               | 1.135  |    1.063     |     1.206     |
|        Contrast

,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
SessionID,0.126,0.032,0.064,0.189,0.000,0.000,5259.0,3590.0,1.0
ContrastHeterogeneity,-7.158,0.482,-8.141,-6.253,0.009,0.009,3259.0,3649.0,1.0
GridCoarseness,-3.785,0.237,-4.244,-3.311,0.004,0.004,3674.0,2964.0,1.0
SessionID:ContrastHeterogeneity,-0.136,0.013,-0.160,-0.111,0.000,0.000,8815.0,6091.0,1.0
SessionID:GridCoarseness,-0.005,0.023,-0.050,0.040,0.000,0.000,5755.0,4385.0,1.0
ContrastHeterogeneity:GridCoarseness,4.200,0.256,3.692,4.687,0.004,0.006,4314.0,2946.0,1.0


In [ ]:
def session_simple_effects_CH(idata, sessions, hdi=0.95):
    # Extract posterior draws for CH main and CH:Session interaction
    posterior = az.extract(idata, var_names=["ContrastHeterogeneity",
                                        "SessionID:ContrastHeterogeneity"]).to_dataframe()
    beta_contrast_heterogeneity   = posterior["ContrastHeterogeneity"].to_numpy()
    beta_session_ch_interaction = posterior["SessionID:ContrastHeterogeneity"].to_numpy()

    rows = []
    for session in sessions:
        beta_draws = beta_contrast_heterogeneity + beta_session_ch_interaction  * session
        odds_ratio_draws   = np.exp(beta_draws)

        hdi_low, hdi_high = az.hdi(beta_draws, hdi_prob=hdi)
        odds_ratio_low, odds_ratio_high = az.hdi(odds_ratio_draws, hdi_prob=hdi)

        rows.append({
            "Session": session,
            "beta_mean": float(beta_draws.mean()),
            f"beta_hdi_{int((1-hdi)/2*100)}%": float(hdi_low),
            f"beta_hdi_{int((1+hdi)/2*100)}%": float(hdi_high),
            "OR_mean": float(odds_ratio_draws.mean()),
            f"OR_hdi_{int((1-hdi)/2*100)}%": float(odds_ratio_low),
            f"OR_hdi_{int((1+hdi)/2*100)}%": float(odds_ratio_high),
            "Pr_beta_less_0": float((beta_draws < 0).mean())
        })
    return pd.DataFrame(rows)

# Example: sessions 1..8
sessions = list(range(1, 9))
tbl_ch_by_session = session_simple_effects_CH(idata_features, sessions, hdi=0.95)
print(tbl_ch_by_session)


   Session  beta_mean  beta_hdi_2%  beta_hdi_97%   OR_mean  OR_hdi_2%  \
0        1  -7.293833    -8.263997     -6.386387  0.000770   0.000196   
1        2  -7.430046    -8.410234     -6.535753  0.000672   0.000166   
2        3  -7.566259    -8.522559     -6.656934  0.000586   0.000143   
3        4  -7.702472    -8.656554     -6.791785  0.000511   0.000122   
4        5  -7.838685    -8.793335     -6.926476  0.000446   0.000105   
5        6  -7.974898    -8.925605     -7.055919  0.000389   0.000099   
6        7  -8.111111    -9.031632     -7.157665  0.000340   0.000077   
7        8  -8.247324    -9.217017     -7.337810  0.000297   0.000070   

   OR_hdi_97%  Pr_beta_less_0  
0    0.001532             1.0  
1    0.001331             1.0  
2    0.001160             1.0  
3    0.001013             1.0  
4    0.000881             1.0  
5    0.000777             1.0  
6    0.000669             1.0  
7    0.000590             1.0  


### Transfer session analysis

In [ ]:
model_transfer = bmb.Model(
    "Correct ~ 1 + ContrastHeterogeneity * GridCoarseness + Transfer + SessionID * (ContrastHeterogeneity + GridCoarseness) +  (1 + SessionID + ContrastHeterogeneity * GridCoarseness|SubjectID)",
    data=data,
    family="bernoulli"
)

In [ ]:
idata_transfer = model_transfer.fit(
    draws=2000, tune=2000, target_accept=0.9,
    idata_kwargs={"log_likelihood": True}, progressbar=False
)

P(Session 9 < Session 2) = 0.009
Δ (S9 − S2) mean = 0.016, 95% CrI [0.002, 0.030]
